# Control Matrix Asset Allocation Optimisation

Optimises a 2D control-matrix policy: allocation is a bilinearly interpolated function
of both time and wealth, parameterised on a grid of `TIME_NODE_COUNT x WEALTH_NODE_COUNT`.

This notebook uses:
- the same maximum time-node count as the time-based case,
- the same wealth-node construction as the wealth-based case,
- distributed initial wealth buckets (not single fixed initial wealth).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import sys
import math

np.random.seed(42)
torch.manual_seed(42)

sys.path.append('..')

from utils.experiment import (
    SimulationConfig,
    OptimizationResult,
    save_experiment,
    load_experiments,
)

from utils import (
    CholeskyBootstrapReturns,
    BlockBootstrapReturnsLoader,
    ControlMatrixPolicy,
    SigmoidWealthPenalty,
    simulate_wealth_trajectory,
    project_onto_simplex,
)
from utils.spending import (
    EXPENDITURE_GUIDELINES,
    DecliningRealSpending,
    DecliningRealFloor,
    NZSuper,
    SpendingPolicy,
)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")

GPU: NVIDIA GeForce RTX 3070
GPU Memory: 8.6 GB


## Configuration Parameters

In [ ]:
# ============================================================================
# SIMULATION PARAMETERS
# ============================================================================
N_ASSETS = 3
N_SIMULATIONS = 400_000
SIMULATION_YEARS = 35

# ============================================================================
# CONTROL MATRIX STRUCTURE
# ============================================================================
# Node-count sweep requested for control-matrix optimisation.
TIME_NODE_COUNTS = [2, 4, 8]
WEALTH_NODE_COUNTS = [2, 3, 6, 10]

# Wealth-node construction from the wealth-based case.
MIN_WEALTH_NODE = 400_000
MAX_WEALTH_NODE = 1_200_000

# ============================================================================
# RETURN SAMPLER
# ============================================================================
RETURN_SAMPLERS = [
    "cholesky",
    "block_bootstrapped",
    "block_bootstrapped_1950",
]

# ============================================================================
# WEALTH & SPENDING PARAMETERS
# ============================================================================
# Distributed initial wealth buckets [500_000, 550_000, ..., 1_000_000].
INITIAL_WEALTH_MIN = 500_000
INITIAL_WEALTH_MAX = 1_000_000
WEALTH_STEP = 50_000

DESIRED_SPENDING = EXPENDITURE_GUIDELINES['choices_metro_couple']
SPENDING_DECLINE_RATE = 0.02
CONSUMPTION_FLOOR = EXPENDITURE_GUIDELINES['no_frills_metro_couple']
FLOOR_DECLINE_RATE = 0.00
INCOME_TYPE = "couple"  # "couple" | "single" | "single_sharing" | None

# ============================================================================
# OPTIMIZATION PARAMETERS
# ============================================================================
# Initial risky-asset weights [bonds_weight, stocks_weight] at each matrix node.
INITIAL_NODE_POLICY = [0.0, 0.0]

OPTIMIZER_TYPE = "SGD"
INITIAL_LR = 5
MIN_LR = 1e-7
MAX_ITERATIONS = 40_000
MOMENTUM = 0.6

# Warmup-Stable-Decay style batch schedule (divisor -> batch size = N_SIMULATIONS / divisor)
# Progresses from noisier small batches to full-batch refinement.
BATCH_DIVISORS = [40, 20, 10, 4, 1]

# Event-driven transition patience per phase (no-improvement steps before moving to next divisor).
# Length must equal len(BATCH_DIVISORS) - 1.
PHASE_ADVANCE_PATIENCE = [50, 50, 50, 50]
MAX_PHASE_ITERS = [2000, 1000, 500, 250]
PHASE_LRS = [5, 2, 1, 0.7, 0.1]

WARM_START_NEW_WEALTH_NODES = False  # Whether to warm-start when wealth-node count increases.

# Allocation-movement stagnation threshold for phase transitions (non-final phases).
# Uses the spread of allocations over the most recent PHASE_MOVEMENT_WINDOW steps.
PHASE_MOVEMENT_TOL = 1e-4
PHASE_MOVEMENT_WINDOW = 20

# LR scaling across batch phases:
# "inverse_sqrt" keeps LR highest at smallest batch, then reduces as batch grows.
# Other options available in loop: "sqrt", "linear", "constant".
BATCH_LR_MODE = "inverse_sqrt"

# Robust model-selection score to reduce lucky mini-batch effects.
# Selection score is a rolling average of recent mini-batch costs.
BEST_SCORE_WINDOW = 50
BEST_MIN_IMPROVEMENT = 1e-5

# Learning-rate schedule (within each batch phase)
LR_DECAY_FACTOR = 0.4
LR_PATIENCE = 100
LR_THRESHOLD = 1e-4

# Early stopping (applies only in the final full-batch phase)
STOPPING_PATIENCE = 100

# Progress reporting
PRINT_EVERY = 10
HISTORY_SAVE_FREQUENCY = 20

# ============================================================================
# OBJECTIVE FUNCTION PARAMETERS
# ============================================================================
WEALTH_PENALTY_STEEPNESS = 1e-3

## Load Data

In [3]:
tax_rates = np.loadtxt("../Data/IID Data/Final/tax_rates.csv", delimiter=",", skiprows=1)

return_sampler_loaders = {}
for sampler_name in RETURN_SAMPLERS:
    if sampler_name == "cholesky":
        exp_returns = np.loadtxt("../Data/IID Data/Final/expected_returns.csv", delimiter=",", skiprows=1)
        cov_matrix = np.loadtxt(
            "../Data/IID Data/Final/covariance.csv", delimiter=",", skiprows=1, usecols=range(1, N_ASSETS + 2)
        )
        return_sampler_loaders[sampler_name] = CholeskyBootstrapReturns(exp_returns, cov_matrix)
    elif sampler_name in {"block_bootstrapped", "block_bootstrapped_1950"}:
        return_sampler_loaders[sampler_name] = BlockBootstrapReturnsLoader(
            f"../Data/Returns/Final/{sampler_name}.npy"
        )
    else:
        raise ValueError(f"Unknown RETURN_SAMPLER: {sampler_name}")

## Optimisation

In [4]:
completed_runs = set()
try:
    existing_experiments = load_experiments(results_dir="Results")
    for payload in existing_experiments:
        cfg = payload["config"]
        completed_runs.add((cfg.RETURN_SAMPLER, int(cfg.TIME_NODE_COUNT), int(cfg.WEALTH_NODE_COUNT)))
    print(f"Loaded {len(completed_runs)} completed runs from Results/.")
except FileNotFoundError:
    print("Results/ not found yet - all runs will be executed.")
except Exception as exc:
    print(f"Could not read existing Results ({exc}); proceeding without skip cache.")

Loaded 3 completed runs from Results/.


In [5]:
from collections import deque
from IPython.display import clear_output
from wakepy import keep

wealth_penalty = SigmoidWealthPenalty(steepness=WEALTH_PENALTY_STEEPNESS)

if len(BATCH_DIVISORS) == 0:
    raise ValueError("BATCH_DIVISORS cannot be empty")
if len(PHASE_ADVANCE_PATIENCE) != max(len(BATCH_DIVISORS) - 1, 0):
    raise ValueError("PHASE_ADVANCE_PATIENCE must have len(BATCH_DIVISORS) - 1 entries")
if len(MAX_PHASE_ITERS) != max(len(BATCH_DIVISORS) - 1, 0):
    raise ValueError("MAX_PHASE_ITERS must have len(BATCH_DIVISORS) - 1 entries")
if len(PHASE_LRS) != len(BATCH_DIVISORS):
    raise ValueError("PHASE_LRS must have len(BATCH_DIVISORS) entries")
if BEST_SCORE_WINDOW < 1:
    raise ValueError("BEST_SCORE_WINDOW must be >= 1")
if BEST_MIN_IMPROVEMENT < 0:
    raise ValueError("BEST_MIN_IMPROVEMENT must be >= 0")
if PHASE_MOVEMENT_TOL < 0:
    raise ValueError("PHASE_MOVEMENT_TOL must be >= 0")
if PHASE_MOVEMENT_WINDOW < 2:
    raise ValueError("PHASE_MOVEMENT_WINDOW must be >= 2")

def reset_sgd_momentum(optimizer: torch.optim.Optimizer, optimizer_type: str) -> None:
    """Clear SGD momentum buffers so new phases don't inherit stale velocity."""
    if optimizer_type.upper() != "SGD":
        return
    for state in optimizer.state.values():
        if "momentum_buffer" in state:
            state["momentum_buffer"].zero_()


def project_policy_inplace(policy: torch.Tensor) -> None:
    with torch.no_grad():
        policy.data = project_onto_simplex(policy.data).clamp(0, 1)


def restore_best_policy_inplace(policy: torch.Tensor, best_policy: torch.Tensor) -> None:
    with torch.no_grad():
        policy.copy_(best_policy.to(policy.device))

def _interpolate_policy_2d(
    src_policy: np.ndarray,
    src_time_nodes: np.ndarray,
    src_wealth_nodes: np.ndarray,
    dst_time_nodes: np.ndarray,
    dst_wealth_nodes: np.ndarray,
) -> np.ndarray:
    """Interpolate (time x wealth x risky_assets) policy surface to a new grid."""
    src_policy = np.asarray(src_policy, dtype=np.float64)
    src_time_nodes = np.asarray(src_time_nodes, dtype=np.float64)
    src_wealth_nodes = np.asarray(src_wealth_nodes, dtype=np.float64)
    dst_time_nodes = np.asarray(dst_time_nodes, dtype=np.float64)
    dst_wealth_nodes = np.asarray(dst_wealth_nodes, dtype=np.float64)

    n_risky = src_policy.shape[2]
    out = np.zeros((len(dst_time_nodes), len(dst_wealth_nodes), n_risky), dtype=np.float64)

    for a in range(n_risky):
        # First interpolate along wealth at each source time node.
        wealth_interp = np.vstack([
            np.interp(dst_wealth_nodes, src_wealth_nodes, src_policy[t_idx, :, a])
            for t_idx in range(len(src_time_nodes))
        ])

        # Then interpolate along time for each destination wealth node.
        for w_idx in range(len(dst_wealth_nodes)):
            out[:, w_idx, a] = np.interp(
                dst_time_nodes,
                src_time_nodes,
                wealth_interp[:, w_idx],
            )

    return out.astype(np.float32)

def _choose_warm_start_key(
    sampler_name: str,
    target_t_count: int,
    target_w_count: int,
    cache: dict,
):
    candidates = []
    for key in cache.keys():
        k_sampler, k_t, k_w = key
        if k_sampler != sampler_name:
            continue
        if k_t > target_t_count or k_w > target_w_count:
            continue
        if k_t == target_t_count and k_w == target_w_count:
            continue
        candidates.append(key)

    if not candidates:
        return None

    # Prefer the highest-resolution prior surface below (or equal to) target.
    return max(candidates, key=lambda k: (k[1] * k[2], k[1], k[2]))

# Distributed initial wealth tensor with near-equal counts per bucket.
wealth_levels = torch.arange(
    INITIAL_WEALTH_MIN, INITIAL_WEALTH_MAX + WEALTH_STEP, WEALTH_STEP, dtype=torch.float32
)
n_buckets = len(wealth_levels)
repeats = N_SIMULATIONS // n_buckets
remainder = N_SIMULATIONS % n_buckets
initial_wealth_tensor = torch.cat([
    wealth_levels.repeat(repeats),
    wealth_levels[:remainder],
]).to(DEVICE)

# Warm-start cache seeded from existing control-matrix results.
warm_start_cache = {}
try:
    existing_experiments = load_experiments(results_dir="Results")
    for payload in existing_experiments:
        cfg = payload["config"]
        res = payload["result"]

        t_count = int(cfg.TIME_NODE_COUNT)
        w_count = int(cfg.WEALTH_NODE_COUNT)
        if t_count <= 1 or w_count <= 1:
            continue

        if res.time_nodes is None or res.wealth_nodes is None:
            continue

        cache_key = (cfg.RETURN_SAMPLER, t_count, w_count)
        warm_start_cache[cache_key] = {
            "policy": np.asarray(res.best_policy, dtype=np.float32),
            "time_nodes": np.asarray(res.time_nodes, dtype=np.float64),
            "wealth_nodes": np.asarray(res.wealth_nodes, dtype=np.float64),
        }
except Exception as exc:
    print(f"Warm-start cache seed skipped ({exc}).")

print(f"Initial wealth buckets: {wealth_levels.numpy()} ({n_buckets} levels, ~{repeats:,} sims each)")
print(f"Batch divisors: {BATCH_DIVISORS}")
print(f"Phase LRs: {PHASE_LRS}")
print(f"Phase advance patience: {PHASE_ADVANCE_PATIENCE}")
print(f"Max phase iterations: {MAX_PHASE_ITERS}")
print(f"Phase movement tolerance: {PHASE_MOVEMENT_TOL}")
print(f"Phase movement window: {PHASE_MOVEMENT_WINDOW}")
print(f"Best-score window: {BEST_SCORE_WINDOW}")
# print(f"Best-score min improvement: {BEST_MIN_IMPROVEMENT}")
print(f"Warm-start cache entries: {len(warm_start_cache)}")

new_wealth_node_count = True
with keep.running():
    for RETURN_SAMPLER in RETURN_SAMPLERS:
        sampler = return_sampler_loaders[RETURN_SAMPLER]
        returns_sample, cumulative_inflation_sample = sampler.generate(N_SIMULATIONS, SIMULATION_YEARS)


        for WEALTH_NODE_COUNT in WEALTH_NODE_COUNTS:
            # Same wealth-node construction as wealth-based case.
            wealth_nodes = torch.logspace(
                math.log10(MIN_WEALTH_NODE), math.log10(MAX_WEALTH_NODE), WEALTH_NODE_COUNT
            )
            new_wealth_node_count = True
            for TIME_NODE_COUNT in TIME_NODE_COUNTS:
                # Same time-node construction as time-based case.
                time_nodes = torch.linspace(0, SIMULATION_YEARS, TIME_NODE_COUNT)

                run_key = (RETURN_SAMPLER, int(TIME_NODE_COUNT), int(WEALTH_NODE_COUNT))
                if run_key in completed_runs:
                    print(
                        f"Skipping completed run: sampler={RETURN_SAMPLER}, "
                        f"TIME_NODE_COUNT={TIME_NODE_COUNT}, WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                    )
                    continue

                print(f"\n{'='*70}")
                print(
                    f"STARTING COMBINATION: sampler={RETURN_SAMPLER}, "
                    f"TIME_NODE_COUNT={TIME_NODE_COUNT}, WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                )
                print(f"Time nodes: {time_nodes.numpy()}")
                print(f"Wealth nodes: {wealth_nodes.numpy()}")
                print(f"{'='*70}")

                returns = torch.tensor(returns_sample - tax_rates, device=DEVICE)
                cumulative_inflation = torch.tensor(cumulative_inflation_sample, device=DEVICE)

                sim_config = SimulationConfig(
                    N_ASSETS=N_ASSETS,
                    N_SIMULATIONS=N_SIMULATIONS,
                    SIMULATION_YEARS=SIMULATION_YEARS,
                    RETURN_SAMPLER=RETURN_SAMPLER,
                    INITIAL_WEALTH=None,  # Distributed bucketed initial wealth
                    DESIRED_SPENDING=DESIRED_SPENDING,
                    SPENDING_DECLINE_RATE=SPENDING_DECLINE_RATE,
                    CONSUMPTION_FLOOR=CONSUMPTION_FLOOR,
                    FLOOR_DECLINE_RATE=FLOOR_DECLINE_RATE,
                    INCOME_TYPE=INCOME_TYPE,
                    INITIAL_POLICY=INITIAL_NODE_POLICY,
                    OPTIMIZER_TYPE=OPTIMIZER_TYPE,
                    TIME_NODE_COUNT=TIME_NODE_COUNT,
                    WEALTH_NODE_COUNT=WEALTH_NODE_COUNT,
                )

                desired_spending_pol = DecliningRealSpending(DESIRED_SPENDING, decline_rate=SPENDING_DECLINE_RATE)
                consumption_floor_pol = DecliningRealFloor(init_floor=CONSUMPTION_FLOOR, decline_rate=FLOOR_DECLINE_RATE)
                income = NZSuper(INCOME_TYPE)
                spending_policy = SpendingPolicy(
                    spending=desired_spending_pol,
                    floor=consumption_floor_pol,
                    income=income,
                )

                allocation_policy = ControlMatrixPolicy(
                    N_ASSETS,
                    N_SIMULATIONS,
                    time_nodes.clone(),
                    wealth_nodes.clone(),
                    DEVICE,
                )

                # Policy tensor shape: (TIME_NODE_COUNT, WEALTH_NODE_COUNT, N_ASSETS-1)
                warm_key = _choose_warm_start_key(
                    RETURN_SAMPLER,
                    int(TIME_NODE_COUNT),
                    int(WEALTH_NODE_COUNT),
                    warm_start_cache,
                )
                
                if new_wealth_node_count and not WARM_START_NEW_WEALTH_NODES:
                    warm_key = None
                    new_wealth_node_count = False

                if warm_key is not None:
                    warm_payload = warm_start_cache[warm_key]
                    warm_policy_np = _interpolate_policy_2d(
                        src_policy=warm_payload["policy"],
                        src_time_nodes=warm_payload["time_nodes"],
                        src_wealth_nodes=warm_payload["wealth_nodes"],
                        dst_time_nodes=time_nodes.detach().cpu().numpy(),
                        dst_wealth_nodes=wealth_nodes.detach().cpu().numpy(),
                    )
                    policy = torch.tensor(
                        warm_policy_np,
                        device=DEVICE,
                        dtype=torch.float32,
                        requires_grad=True,
                    )
                    project_policy_inplace(policy)

                    print(
                        f"Warm start: sampler={RETURN_SAMPLER} using "
                        f"{warm_key[1]}x{warm_key[2]} -> {TIME_NODE_COUNT}x{WEALTH_NODE_COUNT} interpolation"
                    )
                else:
                    base_node = torch.tensor(INITIAL_NODE_POLICY, device=DEVICE, dtype=torch.float32)
                    policy = (
                        base_node
                        .unsqueeze(0)
                        .unsqueeze(0)
                        .repeat(TIME_NODE_COUNT, WEALTH_NODE_COUNT, 1)
                        .requires_grad_(True)
                    )

                initial_phase_lr = PHASE_LRS[0]
                if OPTIMIZER_TYPE.upper() == "ADAM":
                    optimizer = torch.optim.Adam([policy], lr=initial_phase_lr)
                else:
                    optimizer = torch.optim.SGD([policy], lr=initial_phase_lr, momentum=MOMENTUM)

                cost_history = []
                policy_history = []
                best_cost = float('inf')
                best_score = float('inf')
                best_policy = policy.data.clone()
                iterations_without_improvement = 0
                iterations_without_lr_improvement = 0
                iterations_without_phase_improvement = 0

                current_phase_idx = 0
                current_batch_divisor = BATCH_DIVISORS[current_phase_idx]
                current_batch_size = max(1, N_SIMULATIONS // current_batch_divisor)
                phase_iter_count = 0
                recent_policies_for_movement = deque(maxlen=PHASE_MOVEMENT_WINDOW)
                recent_policies_for_movement.append(policy.detach().clone())
                max_policy_window_span = float('inf')

                for i in range(MAX_ITERATIONS):
                    optimizer.zero_grad()
                    phase_iter_count += 1

                    if current_batch_size < N_SIMULATIONS:
                        batch_idx = torch.randint(0, N_SIMULATIONS, (current_batch_size,), device=DEVICE)
                        returns_batch = returns.index_select(0, batch_idx)
                        inflation_batch = cumulative_inflation.index_select(0, batch_idx)
                        wealth_batch = initial_wealth_tensor.index_select(0, batch_idx)
                    else:
                        returns_batch = returns
                        inflation_batch = cumulative_inflation
                        wealth_batch = initial_wealth_tensor

                    wealth, consumption = simulate_wealth_trajectory(
                        returns=returns_batch,
                        cumulative_inflation=inflation_batch,
                        allocation_policy=allocation_policy,
                        spending_policy=spending_policy,
                        initial_wealth=wealth_batch,
                        policy_settings=policy,
                    )

                    cost = wealth_penalty.evaluate(wealth, consumption)
                    cost.backward()
                    optimizer.step()

                    project_policy_inplace(policy)

                    recent_policies_for_movement.append(policy.detach().clone())
                    if len(recent_policies_for_movement) >= 2:
                        movement_stack = torch.stack(tuple(recent_policies_for_movement), dim=0)
                        movement_span = movement_stack.max(dim=0).values - movement_stack.min(dim=0).values
                        max_policy_window_span = float(movement_span.max().item())
                    else:
                        max_policy_window_span = float('inf')

                    cost_item = cost.item()
                    cost_history.append(cost_item)

                    if (i + 1) % HISTORY_SAVE_FREQUENCY == 0:
                        policy_history.append(policy.data.cpu().numpy().copy())

                    # Robust selection score: rolling average over recent mini-batches.
                    best_window_len = min(BEST_SCORE_WINDOW, len(cost_history))
                    this_window_len = min(5, len(cost_history))

                    selection_score = float(np.mean(cost_history[-this_window_len:]))
                    this_best_score = best_score

                    if selection_score < this_best_score:
                        best_score = selection_score
                        best_cost = cost_item
                        best_policy = policy.detach().clone().cpu()
                        iterations_without_improvement = 0
                        iterations_without_lr_improvement = 0
                    else:
                        iterations_without_improvement += 1
                        iterations_without_lr_improvement += 1

                    # In non-final phases, stagnation is defined by small policy spread across recent steps.
                    if current_phase_idx < len(BATCH_DIVISORS) - 1:
                        window_is_full = len(recent_policies_for_movement) == PHASE_MOVEMENT_WINDOW
                        if window_is_full and max_policy_window_span <= PHASE_MOVEMENT_TOL:
                            iterations_without_phase_improvement += 1
                        else:
                            iterations_without_phase_improvement = 0

                    # Cost-based early stopping is only active in final full-batch phase.
                    if current_phase_idx == len(BATCH_DIVISORS) - 1 and iterations_without_improvement >= STOPPING_PATIENCE:
                        print(f"\n{'='*70}")
                        print(f"EARLY STOPPING at iteration {i + 1}")
                        print(f"No final-phase cost improvement in {STOPPING_PATIENCE} iterations")
                        print(f"{'='*70}")
                        break

                    if current_phase_idx < len(BATCH_DIVISORS) - 1:
                        phase_patience = PHASE_ADVANCE_PATIENCE[current_phase_idx]
                        phase_max_iters = MAX_PHASE_ITERS[current_phase_idx]
                        transition_due_to_movement = iterations_without_phase_improvement >= phase_patience
                        transition_due_to_max_iters = phase_iter_count >= phase_max_iters

                        if transition_due_to_movement or transition_due_to_max_iters:
                            old_divisor = current_batch_divisor
                            transition_reason = (
                                f"movement stagnation ({iterations_without_phase_improvement}/{phase_patience})"
                                if transition_due_to_movement
                                else f"max phase iterations reached ({phase_iter_count}/{phase_max_iters})"
                            )

                            current_phase_idx += 1
                            current_batch_divisor = BATCH_DIVISORS[current_phase_idx]
                            current_batch_size = max(1, N_SIMULATIONS // current_batch_divisor)
                            target_phase_lr = PHASE_LRS[current_phase_idx]

                            for param_group in optimizer.param_groups:
                                param_group['lr'] = target_phase_lr

                            iterations_without_phase_improvement = 0
                            iterations_without_lr_improvement = 0
                            iterations_without_improvement = 0
                            phase_iter_count = 0

                            # Keep current policy when moving between phases.
                            recent_policies_for_movement.clear()
                            recent_policies_for_movement.append(policy.detach().clone())
                            max_policy_window_span = float('inf')
                            reset_sgd_momentum(optimizer, OPTIMIZER_TYPE)

                            print(
                                f"Phase transition at iter {i+1}: "
                                f"batch 1/{old_divisor} -> 1/{current_batch_divisor}, "
                                f"LR set to {target_phase_lr:.6f}, reason={transition_reason}"
                            )

                            if current_phase_idx == len(BATCH_DIVISORS) - 1:
                                # Entering final full-batch phase: reset best metrics on full-batch objective.
                                with torch.no_grad():
                                    full_batch_wealth, full_batch_consumption = simulate_wealth_trajectory(
                                        returns=returns,
                                        cumulative_inflation=cumulative_inflation,
                                        allocation_policy=allocation_policy,
                                        spending_policy=spending_policy,
                                        initial_wealth=initial_wealth_tensor,
                                        policy_settings=policy,
                                    )
                                    full_batch_cost = float(
                                        wealth_penalty.evaluate(full_batch_wealth, full_batch_consumption).item()
                                    )

                                best_score = full_batch_cost
                                best_cost = full_batch_cost
                                best_policy = policy.detach().clone().cpu()
                                iterations_without_improvement = 0
                                iterations_without_lr_improvement = 0
                                iterations_without_phase_improvement = 0
                                print(f"Final phase baseline reset: best_cost={best_cost:.6f}")

                    if iterations_without_lr_improvement >= LR_PATIENCE:
                        # LR decay is only applied in the final (full-batch) phase.
                        if current_phase_idx < len(BATCH_DIVISORS) - 1:
                            iterations_without_lr_improvement = 0
                        else:
                            stop = False
                            print(
                                f"\n{'='*70}\n"
                                f"LR decay at iteration {i + 1} due to no improvement in {LR_PATIENCE} iterations\n"
                                f"Current phase: final phase\n"
                                f"Batch divisor: 1/{current_batch_divisor}\n"
                                f"{'='*70}"
                            )

                            old_lr = optimizer.param_groups[0]['lr']
                            new_lr = old_lr
                            for param_group in optimizer.param_groups:
                                new_lr = param_group['lr'] * LR_DECAY_FACTOR
                                param_group['lr'] = new_lr
                                if new_lr < MIN_LR:
                                    print(f"\n{'='*70}")
                                    print(f"LR below minimum threshold at iteration {i + 1}: {new_lr:.6f} < {MIN_LR:.6f}")
                                    print("Stopping optimization.")
                                    print(f"{'='*70}")
                                    stop = True
                                    break
                            if stop:
                                break

                            print(f"LR reduced at iteration {i+1}: {old_lr:.6f} -> {new_lr:.6f}")
                            iterations_without_lr_improvement = 0

                            restore_best_policy_inplace(policy, best_policy)
                            recent_policies_for_movement.clear()
                            recent_policies_for_movement.append(policy.detach().clone())
                            max_policy_window_span = float('inf')

                            # Reset optimizer state so reduced-LR phase starts without stale momentum.
                            reset_sgd_momentum(optimizer, OPTIMIZER_TYPE)

                    if (i + 1) % PRINT_EVERY == 0 or i == 0:
                        clear_output(wait=True)
                        current_lr = optimizer.param_groups[0]['lr']
                        cost_change = cost_history[-1] - cost_history[-2] if len(cost_history) > 1 else 0

                        phase_stagnation = (
                            iterations_without_phase_improvement
                            if current_phase_idx < len(BATCH_DIVISORS) - 1
                            else 0
                        )
                        phase_patience_text = (
                            str(PHASE_ADVANCE_PATIENCE[current_phase_idx])
                            if current_phase_idx < len(BATCH_DIVISORS) - 1
                            else "final"
                        )
                        phase_iter_cap_text = (
                            str(MAX_PHASE_ITERS[current_phase_idx])
                            if current_phase_idx < len(BATCH_DIVISORS) - 1
                            else "final"
                        )
                        movement_window_fill = len(recent_policies_for_movement)
                        movement_span_text = (
                            f"{max_policy_window_span:.6e}" if movement_window_fill >= 2 else "n/a"
                        )

                        print(f"{'='*70}")
                        print(
                            f"OPTIMISING: sampler={RETURN_SAMPLER} | "
                            f"TIME_NODE_COUNT={TIME_NODE_COUNT} | WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                        )
                        print(
                            f"Iteration {i+1:,}/{MAX_ITERATIONS:,} ({(i+1)/MAX_ITERATIONS*100:.1f}%)  |  "
                            f"LR: {current_lr:.6f} | phase={current_phase_idx+1}/{len(BATCH_DIVISORS)} | "
                            f"batch=1/{current_batch_divisor} ({current_batch_size:,} sims)"
                        )
                        print(
                            f"Phase stagnation (movement): {phase_stagnation}/{phase_patience_text} "
                            f"(tol={PHASE_MOVEMENT_TOL:.2e})"
                        )
                        print(f"Phase iterations:          {phase_iter_count}/{phase_iter_cap_text}")
                        print(
                            f"Movement window fill:       {movement_window_fill}/{PHASE_MOVEMENT_WINDOW}"
                        )
                        print(f"Max policy span (window):   {movement_span_text}")
                        print(f"{'='*70}")
                        print(f"Cost:                      {cost_item:.6f}")
                        print(f"Selection score:           {selection_score:.6f} (window={best_window_len})")
                        print(f"Cost change:               {cost_change:.8f}")
                        print(f"Best selection score:      {best_score:.6f}")
                        print(f"Best cost at selection:    {best_cost:.6f}")
                        print(f"Final-phase w/o improve:   {iterations_without_improvement}/{STOPPING_PATIENCE}")
                        print(f"{'-'*70}")
                        mid_w_idx = WEALTH_NODE_COUNT // 2
                        for t_idx in range(TIME_NODE_COUNT):
                            age = int(time_nodes[t_idx].item()) + 65
                            low = policy.data[t_idx, 0].cpu().numpy()
                            mid = policy.data[t_idx, mid_w_idx].cpu().numpy()
                            high = policy.data[t_idx, -1].cpu().numpy()
                            print(
                                f"  Time node {t_idx} (age {age}): "
                                f"lowW bonds/stocks={low[0]:.1%}/{low[1]:.1%}, "
                                f"midW={mid[0]:.1%}/{mid[1]:.1%}, "
                                f"highW={high[0]:.1%}/{high[1]:.1%}"
                            )
                        print(f"{'-'*70}")
                        print(f"Mean consumption:          ${consumption.mean().item():,.0f}")
                        print(f"Mean terminal wealth:      ${wealth[:, -1].mean().item():,.0f}")
                        print(f"Bankruptcy rate:           {(wealth[:, -1] == 0).sum().item() / wealth.shape[0]:.2%}")
                        bankruptcy_density = (wealth == 0).sum().item() / (wealth.shape[0] * SIMULATION_YEARS)
                        floor_t = consumption_floor_pol.calculate_tensor(
                            wealth=wealth, cumulative_inflation=inflation_batch
                        )
                        impoverishment_density = (consumption < floor_t).sum().item() / (wealth.shape[0] * SIMULATION_YEARS)
                        print(f"Impoverishment density:    {impoverishment_density:.4%}")
                        print(f"Bankruptcy density:        {bankruptcy_density:.4%}")
                        print(f"{'='*70}")

                restore_best_policy_inplace(policy, best_policy)

                policy_history = np.array(policy_history)
                cost_history = np.array(cost_history)

                print(f"\n{'='*70}")
                print(
                    f"OPTIMISATION COMPLETE - sampler={RETURN_SAMPLER} "
                    f"| TIME_NODE_COUNT={TIME_NODE_COUNT} | WEALTH_NODE_COUNT={WEALTH_NODE_COUNT}"
                )
                print(f"Best selection score: {best_score:.6f}  |  Iterations: {len(cost_history):,}")
                print(f"{'='*70}")

                # Final evaluation saved on full simulation set.
                final_wealth, final_consumption = simulate_wealth_trajectory(
                    returns=returns,
                    cumulative_inflation=cumulative_inflation,
                    allocation_policy=allocation_policy,
                    spending_policy=spending_policy,
                    initial_wealth=initial_wealth_tensor,
                    policy_settings=policy,
                )

                result = OptimizationResult(
                    best_policy=best_policy.detach().cpu().numpy(),
                    best_utility=-best_score,
                    policy_history=policy_history,
                    cost_history=cost_history,
                    wealth_simulated=final_wealth.detach().cpu().numpy(),
                    consumption_simulated=final_consumption.detach().cpu().numpy(),
                    cumulative_inflation=cumulative_inflation.detach().cpu().numpy(),
                    time_nodes=time_nodes.detach().cpu().numpy(),
                    wealth_nodes=wealth_nodes.detach().cpu().numpy(),
                )
                save_experiment(sim_config, result)

                # Make this freshly solved surface available as a warm-start source.
                warm_start_cache[(RETURN_SAMPLER, int(TIME_NODE_COUNT), int(WEALTH_NODE_COUNT))] = {
                    "policy": best_policy.detach().cpu().numpy(),
                    "time_nodes": time_nodes.detach().cpu().numpy(),
                    "wealth_nodes": wealth_nodes.detach().cpu().numpy(),
                }
                completed_runs.add(run_key)

OPTIMISING: sampler=cholesky | TIME_NODE_COUNT=2 | WEALTH_NODE_COUNT=3
Iteration 2,660/40,000 (6.7%)  |  LR: 5.000000 | phase=1/5 | batch=1/40 (10,000 sims)
Phase stagnation (movement): 0/50 (tol=1.00e-04)
Phase iterations:          2660/5000
Movement window fill:       20/20
Max policy span (window):   5.165401e-02
Cost:                      -0.938179
Selection score:           -0.938020 (window=50)
Cost change:               -0.00094953
Best selection score:      -0.939921
Best cost at selection:    -0.939520
Final-phase w/o improve:   2369/100
----------------------------------------------------------------------
  Time node 0 (age 65): lowW bonds/stocks=0.0%/100.0%, midW=46.1%/53.9%, highW=0.0%/0.7%
  Time node 1 (age 100): lowW bonds/stocks=0.0%/100.0%, midW=100.0%/0.0%, highW=0.0%/0.0%
----------------------------------------------------------------------
Mean consumption:          $102,644
Mean terminal wealth:      $386,292
Bankruptcy rate:           24.58%
Impoverishment densi

KeyboardInterrupt: 